In [1]:
# Import các thư viện cần thiết
import pandas as pd
import numpy as np
import re
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load dataset
df = pd.read_csv('../data/all_recipes_final.csv')
print(f"Dataset shape: {df.shape}")
df.head()


Dataset shape: (10263, 12)
Phân bố theo nguồn:
source
dienmayxanh    8993
vnexpress       737
vncooking       533
Name: count, dtype: int64


,title,type_of_food,link,description,ingredients,step,note,num_of_ingredients,cook_time,num_of_people,calories,source
0,Cách muối dưa hành truyền thống,Món Tết,https://vnexpress.net/doi-song-cooking-cach-mu...,Dưa hành muối là món ăn truyền thống ngày Tết ...,"['1 kg hành củ tươi', 'Tro bếp hoặc nước vo gọ...",['Bước 1: Chọn hành củ: Nên chọn hành củ ta bá...,[],5,45 phút,8-10 người,459 kcal,vnexpress
1,Su hào xào mực - món cổ Tết Bát Tràng,Món Tết,https://vnexpress.net/doi-song-cooking-su-hao-...,Đĩa xào khô ráo với su hào giòn ngọt quyện với...,"['2 củ su hào non', '1 con mực khô', '1/2 củ c...",['Bước 1: Chọn và sơ chế mực: Người dân làng g...,['Su hào xào mực cùng với canh măng mực là hai...,6,50 phút,4 - 5 người,1.162 kcal,vnexpress
2,Canh măng ngày Tết cổ truyền Hà Nội,Món Tết,https://vnexpress.net/doi-song-cooking-canh-ma...,"Măng ngấu vị, giòn ngon, móng giò hầm vừa độ s...","['800 gr măng khô', '2 móng giò lợn', 'Nước dù...","['Bước 1: Chọn măng khô: Theo lối cũ, người nộ...",['Nếu tận dụng nước luộc gà nấu canh măng thì ...,6,100 phút,8 - 10 người,4.930 kcal,vnexpress
3,Giả hạnh nhân - món ngon Tết xưa Hà Nội,Món Tết,https://vnexpress.net/doi-song-cooking-gia-han...,Đây là món ăn cổ truyền thường thấy trong cỗ T...,"['2 bộ lòng mề gà', '100 gr lạc', '50 gr hạt đ...",['Bước 1: Chọn và sơ chế lạc: Chọn lạc khô chắ...,['Hạnh nhân xào (hay giả hạnh nhân) là món ăn ...,8,60 phút,4-5 người,1.112 kcal,vnexpress
4,Chả bì ớt xiêm xanh,Món Tết,https://vnexpress.net/doi-song-cooking-cha-bi-...,"Chả bì bóng đẹp, gói đều tay. Khi ăn vị ngọt m...","['500 gr giò sống', '300 gr bì lợn', '20 - 30 ...","['Bước 1: Chọn và sơ chế bì lợn, chuẩn bị giò ...",['Nên sơ chế kỹ bì lợn để chả được thơm. Tùy t...,6,60 phút,5-6 người,2.512 kcal,vnexpress


## 1. Data Preprocessing

In [3]:
# Xử lý missing values
data = df.copy()

data['title'] = data['title'].fillna('')
data['description'] = data['description'].fillna('')
data['step'] = data['step'].fillna('[]')
data['ingredients'] = data['ingredients'].fillna('[]')
data['type_of_food'] = data['type_of_food'].fillna('Unknown')

print("Missing values after handling:")
print(data[['title', 'description', 'step', 'ingredients', 'type_of_food', 'calories', 'cook_time']].isnull().sum())


Missing values after handling:
title              0
description        0
step               0
ingredients        0
type_of_food       0
calories        9826
cook_time        295
dtype: int64


In [4]:
# Định nghĩa các hàm xử lý và làm sạch dữ liệu
def parse_list_string(s):
    if pd.isna(s) or s == '[]':
        return []
    try:
        return ast.literal_eval(s)
    except:
        return []

def clean_text(text):
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r'[^\w\s\u00C0-\u1EF9]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def parse_cook_time(time_str):
    if pd.isna(time_str):
        return np.nan
    time_str = str(time_str).lower()
    minutes = 0
    
    hour_match = re.search(r'(\d+)\s*(?:giờ|h|hour)', time_str)
    if hour_match:
        minutes += int(hour_match.group(1)) * 60
    
    min_match = re.search(r'(\d+)\s*(?:phút|p|min|minute)', time_str)
    if min_match:
        minutes += int(min_match.group(1))
    
    if minutes == 0:
        num_match = re.search(r'(\d+)', time_str)
        if num_match:
            minutes = int(num_match.group(1))
    
    return minutes if minutes > 0 else np.nan

def parse_calories(cal_str):
    if pd.isna(cal_str):
        return np.nan
    cal_str = str(cal_str).replace('.', '').replace(',', '')
    match = re.search(r'(\d+)', cal_str)
    if match:
        return float(match.group(1))
    return np.nan


In [5]:
# Áp dụng các hàm preprocessing lên dữ liệu
data['ingredients_list'] = data['ingredients'].apply(parse_list_string)
data['step_list'] = data['step'].apply(parse_list_string)

data['title_clean'] = data['title'].apply(clean_text)
data['description_clean'] = data['description'].apply(clean_text)
data['step_clean'] = data['step_list'].apply(lambda x: ' '.join([clean_text(s) for s in x]))

data['cook_time_minutes'] = data['cook_time'].apply(parse_cook_time)
data['calories_numeric'] = data['calories'].apply(parse_calories)

data['ingredients_clean'] = data['ingredients_list'].apply(
    lambda x: set([clean_text(ing) for ing in x if ing])
)

data[['title', 'title_clean', 'cook_time', 'cook_time_minutes', 'calories', 'calories_numeric']].head()


,title,title_clean,cook_time,cook_time_minutes,calories,calories_numeric
0,Cách muối dưa hành truyền thống,cách muối dưa hành truyền thống,45 phút,45.0,459 kcal,459.0
1,Su hào xào mực - món cổ Tết Bát Tràng,su hào xào mực món cổ tết bát tràng,50 phút,50.0,1.162 kcal,1162.0
2,Canh măng ngày Tết cổ truyền Hà Nội,canh măng ngày tết cổ truyền hà nội,100 phút,100.0,4.930 kcal,4930.0
3,Giả hạnh nhân - món ngon Tết xưa Hà Nội,giả hạnh nhân món ngon tết xưa hà nội,60 phút,60.0,1.112 kcal,1112.0
4,Chả bì ớt xiêm xanh,chả bì ớt xiêm xanh,60 phút,60.0,2.512 kcal,2512.0


In [6]:
# Kiểm tra kết quả preprocessing
print("Sample ingredients_clean:")
for i, ing in enumerate(data['ingredients_clean'].head(3)):
    print(f"\nRecipe {i+1}: {data['title'].iloc[i]}")
    print(f"Ingredients: {ing}")

Sample ingredients_clean:

Recipe 1: Cách muối dưa hành truyền thống
Ingredients: {'tro bếp hoặc nước vo gọa', 'lọ sạch', 'cà rốt trang trí tùy chọn', '1 kg hành củ tươi', 'muối hạt đường'}

Recipe 2: Su hào xào mực - món cổ Tết Bát Tràng
Ingredients: {'gia vị mắm muối đường hạt tiêu rượu trắng gừng', '1 con mực khô', '2 củ su hào non', 'rau mùi trang trí', 'mỡ lợn hoặc dầu ăn', '1 2 củ cà rốt'}

Recipe 3: Canh măng ngày Tết cổ truyền Hà Nội
Ingredients: {'800 gr măng khô', 'nước vo gạo ngâm măng', 'nước dùng gà hoặc ninh xương lợn', '2 móng giò lợn', 'hành khô hành củ', 'gia vị nước mắm truyền thống muối'}


## 2. TF-IDF Based Recommendation

Tính toán similarity dựa trên nội dung văn bản (title, description, steps) sử dụng TF-IDF và cosine similarity.


In [7]:
# Tạo TF-IDF matrix từ text features
data['combined_text'] = data['title_clean'] + ' ' + data['description_clean'] + ' ' + data['step_clean']

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95
)

tfidf_matrix = tfidf_vectorizer.fit_transform(data['combined_text'])
print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")


TF-IDF Matrix shape: (10263, 5000)


In [8]:
# Tính cosine similarity matrix từ TF-IDF
tfidf_similarity = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"TF-IDF Similarity Matrix shape: {tfidf_similarity.shape}")


TF-IDF Similarity Matrix shape: (10263, 10263)


In [9]:
# Hàm lấy recommendations dựa trên TF-IDF
def get_tfidf_recommendations(recipe_idx, similarity_matrix, df, top_n=5):
    sim_scores = list(enumerate(similarity_matrix[recipe_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    recipe_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    result = df.iloc[recipe_indices][['title', 'type_of_food', 'calories', 'cook_time']].copy()
    result['tfidf_score'] = scores
    
    return result

def recommend_by_title_tfidf(title, df, similarity_matrix, top_n=5):
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    print(f"\nInput: {df.loc[recipe_idx, 'title']}")
    print(f"   Type: {df.loc[recipe_idx, 'type_of_food']}, Calories: {df.loc[recipe_idx, 'calories']}")
    print("\nTF-IDF Recommendations:")
    
    return get_tfidf_recommendations(recipe_idx, similarity_matrix, df, top_n)


In [ ]:
# Save TF-IDF model and similarity matrix for evaluation
import pickle
import os

os.makedirs('../notebooks/saved_models', exist_ok=True)

# Save TF-IDF vectorizer
with open('../notebooks/saved_models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)

# Save TF-IDF matrix
with open('../notebooks/saved_models/tfidf_matrix.pkl', 'wb') as f:
    pickle.dump(tfidf_matrix, f)

# Save TF-IDF similarity matrix
np.save('../notebooks/saved_models/tfidf_similarity.npy', tfidf_similarity)

# Save recommendation functions
with open('../notebooks/saved_models/tfidf_recommender_funcs.pkl', 'wb') as f:
    pickle.dump({
        'get_recommendations': get_tfidf_recommendations,
        'recommend_by_title': recommend_by_title_tfidf
    }, f)

print("✓ TF-IDF method saved successfully!")
print(f"  - Vectorizer: tfidf_vectorizer.pkl")
print(f"  - Matrix: tfidf_matrix.pkl")
print(f"  - Similarity: tfidf_similarity.npy")
print(f"  - Functions: tfidf_recommender_funcs.pkl")

## 3. Ingredient TF-IDF Based Recommendation

**Ingredient TF-IDF** xử lý ingredient list như text documents, tốt vì:
- Xử lý được variations trong cách viết (thịt bò, bò, beef...)
- Gán trọng số cho ingredients based on importance
- Không bị ảnh hưởng bởi exact string matching

**Ý tưởng**: Mỗi recipe là 1 document, ingredients là words. Áp dụng TF-IDF để tính similarity.

In [10]:
# Chuẩn bị ingredient text cho TF-IDF
# Join ingredients thành string separated by spaces
data['ingredients_text'] = data['ingredients_list'].apply(
    lambda x: ' '.join([clean_text(ing) for ing in x if ing])
)

print("Sample ingredient texts:")
for i in range(3):
    print(f"\n{data['title'].iloc[i]}")
    print(f"   Ingredients text: {data['ingredients_text'].iloc[i][:100]}...")

# Build TF-IDF vectorizer cho ingredients
print("\nBuilding Ingredient TF-IDF matrix...")
ingredient_tfidf_vectorizer = TfidfVectorizer(
    max_features=2000,  # Fewer features than text-based
    ngram_range=(1, 2),  # Unigrams and bigrams
    min_df=2,
    max_df=0.8
)

ingredient_tfidf_matrix = ingredient_tfidf_vectorizer.fit_transform(data['ingredients_text'])
print(f"Ingredient TF-IDF Matrix shape: {ingredient_tfidf_matrix.shape}")

# Tính cosine similarity
ingredient_tfidf_similarity = cosine_similarity(ingredient_tfidf_matrix, ingredient_tfidf_matrix)
print(f"Ingredient TF-IDF Similarity Matrix shape: {ingredient_tfidf_similarity.shape}")

Sample ingredient texts:

Cách muối dưa hành truyền thống
   Ingredients text: 1 kg hành củ tươi tro bếp hoặc nước vo gọa muối hạt đường cà rốt trang trí tùy chọn lọ sạch...

Su hào xào mực - món cổ Tết Bát Tràng
   Ingredients text: 2 củ su hào non 1 con mực khô 1 2 củ cà rốt gia vị mắm muối đường hạt tiêu rượu trắng gừng rau mùi t...

Canh măng ngày Tết cổ truyền Hà Nội
   Ingredients text: 800 gr măng khô 2 móng giò lợn nước dùng gà hoặc ninh xương lợn hành khô hành củ gia vị nước mắm tru...

Building Ingredient TF-IDF matrix...
Ingredient TF-IDF Matrix shape: (10263, 2000)
Ingredient TF-IDF Similarity Matrix shape: (10263, 10263)


In [11]:
# Hàm lấy recommendations dựa trên Ingredient TF-IDF
def get_ingredient_tfidf_recommendations(recipe_idx, similarity_matrix, df, top_n=5):
    sim_scores = list(enumerate(similarity_matrix[recipe_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    recipe_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    result = df.iloc[recipe_indices][['title', 'type_of_food', 'calories', 'cook_time']].copy()
    result['ing_tfidf_score'] = scores
    
    return result

def recommend_by_title_ingredient_tfidf(title, df, similarity_matrix, top_n=5):
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    print(f"\nInput: {df.loc[recipe_idx, 'title']}")
    print(f"   Ingredients: {df.loc[recipe_idx, 'ingredients_text'][:100]}...")
    print("\nIngredient TF-IDF Recommendations:")
    
    return get_ingredient_tfidf_recommendations(recipe_idx, similarity_matrix, df, top_n)

print("Ingredient TF-IDF recommendation functions ready!")

Ingredient TF-IDF recommendation functions ready!


In [ ]:
# Save Ingredient TF-IDF model and similarity matrix for evaluation
# Save Ingredient TF-IDF vectorizer
with open('../notebooks/saved_models/ingredient_tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(ingredient_tfidf_vectorizer, f)

# Save Ingredient TF-IDF matrix
with open('../notebooks/saved_models/ingredient_tfidf_matrix.pkl', 'wb') as f:
    pickle.dump(ingredient_tfidf_matrix, f)

# Save Ingredient TF-IDF similarity matrix
np.save('../notebooks/saved_models/ingredient_tfidf_similarity.npy', ingredient_tfidf_similarity)

# Save recommendation functions
with open('../notebooks/saved_models/ingredient_tfidf_recommender_funcs.pkl', 'wb') as f:
    pickle.dump({
        'get_recommendations': get_ingredient_tfidf_recommendations,
        'recommend_by_title': recommend_by_title_ingredient_tfidf
    }, f)

print("✓ Ingredient TF-IDF method saved successfully!")
print(f"  - Vectorizer: ingredient_tfidf_vectorizer.pkl")
print(f"  - Matrix: ingredient_tfidf_matrix.pkl")
print(f"  - Similarity: ingredient_tfidf_similarity.npy")
print(f"  - Functions: ingredient_tfidf_recommender_funcs.pkl")

## 4. Hybrid Approach

Kết hợp 2 phương pháp với trọng số: TF-IDF (0.4) + Ingredient TF-IDF (0.6)

**Rationale**: Ingredients quan trọng nhất trong food recommendation, nên tăng Ingredient TF-IDF weight

In [13]:
# Hàm tính hybrid similarity (kết hợp TF-IDF và Ingredient TF-IDF)
def compute_hybrid_similarity(tfidf_sim, ing_tfidf_sim, 
                              w_tfidf=0.4, w_ing_tfidf=0.6):
    total_weight = w_tfidf + w_ing_tfidf
    w_tfidf /= total_weight
    w_ing_tfidf /= total_weight
    
    print(f"Weights: TF-IDF={w_tfidf:.2f}, Ingredient TF-IDF={w_ing_tfidf:.2f}")
    
    hybrid_similarity = (w_tfidf * tfidf_sim + 
                        w_ing_tfidf * ing_tfidf_sim)
    
    return hybrid_similarity

In [14]:
# Tính hybrid similarity matrix
hybrid_similarity_matrix = compute_hybrid_similarity(
    tfidf_similarity, 
    ingredient_tfidf_similarity,
    w_tfidf=0.4,
    w_ing_tfidf=0.6
)
print(f"Hybrid Similarity Matrix shape: {hybrid_similarity_matrix.shape}")

Weights: TF-IDF=0.40, Ingredient TF-IDF=0.60
Hybrid Similarity Matrix shape: (10263, 10263)


In [15]:
# Hàm lấy recommendations dựa trên hybrid approach
def get_hybrid_recommendations(recipe_idx, hybrid_sim, tfidf_sim, ing_tfidf_sim, df, top_n=5):
    sim_scores = list(enumerate(hybrid_sim[recipe_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    recipe_indices = [i[0] for i in sim_scores]
    hybrid_scores = [i[1] for i in sim_scores]
    
    result = df.iloc[recipe_indices][['title', 'type_of_food', 'calories', 'cook_time']].copy()
    result['hybrid_score'] = hybrid_scores
    result['tfidf_score'] = [tfidf_sim[recipe_idx][i] for i in recipe_indices]
    result['ing_tfidf_score'] = [ing_tfidf_sim[recipe_idx][i] for i in recipe_indices]
    
    return result

def recommend_by_title_hybrid(title, df, hybrid_sim, tfidf_sim, ing_tfidf_sim, top_n=5):
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"❌ No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    print(f"\nInput: {df.loc[recipe_idx, 'title']}")
    print(f"   Type: {df.loc[recipe_idx, 'type_of_food']}, Calories: {df.loc[recipe_idx, 'calories']}, Time: {df.loc[recipe_idx, 'cook_time']}")
    print("\nHybrid Recommendations:")
    
    return get_hybrid_recommendations(recipe_idx, hybrid_sim, tfidf_sim, ing_tfidf_sim, df, top_n)

In [ ]:
# Save Hybrid similarity matrix for evaluation
# Save Hybrid similarity matrix
np.save('../notebooks/saved_models/hybrid_similarity.npy', hybrid_similarity_matrix)

# Save recommendation functions
with open('../notebooks/saved_models/hybrid_recommender_funcs.pkl', 'wb') as f:
    pickle.dump({
        'get_recommendations': get_hybrid_recommendations,
        'recommend_by_title': recommend_by_title_hybrid
    }, f)

print("✓ Hybrid method saved successfully!")
print(f"  - Similarity: hybrid_similarity.npy")
print(f"  - Functions: hybrid_recommender_funcs.pkl")

## 5. Keyword-Based Recommendation

Extract keywords chính từ title (nguyên liệu chính + phương pháp nấu).

**Ưu điểm**: Đơn giản, focus vào main keywords, dễ giải thích

In [16]:
# Hàm lấy recommendations dựa trên keywords
def get_keyword_recommendations(recipe_idx, similarity_matrix, df, top_n=5):
    sim_scores = list(enumerate(similarity_matrix[recipe_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    recipe_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    result = df.iloc[recipe_indices][['title', 'type_of_food', 'calories', 'cook_time']].copy()
    result['keyword_score'] = scores
    
    return result

def recommend_by_title_keyword(title, df, similarity_matrix, top_n=5):
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    print(f"\nInput: {df.loc[recipe_idx, 'title']}")
    print(f"   Keywords: {df.loc[recipe_idx, 'keywords']}")
    print("\nKeyword-Based Recommendations:")
    
    return get_keyword_recommendations(recipe_idx, similarity_matrix, df, top_n)

In [17]:
# Hàm extract keywords/tags từ title
def extract_keywords(title):
    """
    Extract main keywords from recipe title
    Focus on: ingredients, cooking methods, dish types
    """
    if pd.isna(title):
        return set()
    
    title = clean_text(title)
    
    # Common Vietnamese cooking methods and dish types
    cooking_methods = ['xào', 'nướng', 'luộc', 'chiên', 'hấp', 'kho', 'rim', 'rang', 
                       'canh', 'súp', 'cháo', 'gỏi', 'nộm', 'salad', 'bún', 'phở', 
                       'mì', 'cơm', 'bánh', 'chè', 'sinh tố']
    
    # Common ingredients
    main_ingredients = ['thịt', 'bò', 'gà', 'heo', 'lợn', 'cá', 'tôm', 'mực', 'nghêu',
                        'rau', 'củ', 'quả', 'trứng', 'đậu', 'nấm', 'măng', 'bí', 
                        'cà', 'khoai', 'su', 'hào', 'cải', 'rau muống', 'rau cần']
    
    # Extract keywords
    keywords = set()
    words = title.split()
    
    # Add cooking methods
    for method in cooking_methods:
        if method in title:
            keywords.add(method)
    
    # Add ingredients (check for 2-word and 1-word matches)
    for i in range(len(words)):
        # Check 2-word combinations
        if i < len(words) - 1:
            two_word = f"{words[i]} {words[i+1]}"
            for ing in main_ingredients:
                if ing in two_word:
                    keywords.add(ing)
        
        # Check single words
        for ing in main_ingredients:
            if ing in words[i]:
                keywords.add(ing)
    
    # Add all significant words (length > 2) as backup
    for word in words:
        if len(word) > 2:
            keywords.add(word)
    
    return keywords

# Apply keyword extraction to all recipes
data['keywords'] = data['title'].apply(extract_keywords)
print(f"Keyword extraction completed!")

# Show examples
print("\nSample keywords:")
for i in range(5):
    print(f"\n{data['title'].iloc[i]}")
    print(f"   Keywords: {data['keywords'].iloc[i]}")

Keyword extraction completed!

Sample keywords:

Cách muối dưa hành truyền thống
   Keywords: {'truyền', 'cá', 'cách', 'muối', 'dưa', 'hành', 'thống'}

Su hào xào mực - món cổ Tết Bát Tràng
   Keywords: {'hào', 'bát', 'tết', 'su', 'xào', 'mực', 'tràng', 'món'}

Canh măng ngày Tết cổ truyền Hà Nội
   Keywords: {'truyền', 'măng', 'tết', 'nội', 'canh', 'ngày', 'gà'}

Giả hạnh nhân - món ngon Tết xưa Hà Nội
   Keywords: {'nhân', 'ngon', 'tết', 'giả', 'nội', 'hạnh', 'món', 'xưa'}

Chả bì ớt xiêm xanh
   Keywords: {'xanh', 'xiêm', 'chả'}


In [18]:
# Helper functions cho Keyword similarity (sử dụng Jaccard cho keyword sets)
def jaccard_similarity(set1, set2):
    """
    Tính Jaccard similarity giữa 2 sets
    J(A,B) = |A ∩ B| / |A ∪ B|
    """
    if len(set1) == 0 and len(set2) == 0:
        return 0.0
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union if union > 0 else 0.0

def compute_jaccard_similarity_matrix(items_list):
    """
    Tính Jaccard similarity matrix cho list of sets
    Sử dụng cho keyword-based recommendation
    """
    n = len(items_list)
    similarity_matrix = np.zeros((n, n))
    
    for i in range(n):
        for j in range(i, n):
            sim = jaccard_similarity(items_list[i], items_list[j])
            similarity_matrix[i][j] = sim
            similarity_matrix[j][i] = sim
    
    return similarity_matrix

print("Jaccard helper functions defined (for keyword similarity only)!")

Jaccard helper functions defined (for keyword similarity only)!


In [19]:
# Tính Keyword-based similarity matrix
keywords_sets = data['keywords'].tolist()
keyword_similarity_matrix = compute_jaccard_similarity_matrix(keywords_sets)
print(f"Keyword Similarity Matrix shape: {keyword_similarity_matrix.shape}")

Keyword Similarity Matrix shape: (10263, 10263)


In [ ]:
# Save Keyword-based similarity matrix for evaluation
# Save Keyword similarity matrix
np.save('../notebooks/saved_models/keyword_similarity.npy', keyword_similarity_matrix)

# Save recommendation functions
with open('../notebooks/saved_models/keyword_recommender_funcs.pkl', 'wb') as f:
    pickle.dump({
        'get_recommendations': get_keyword_recommendations,
        'recommend_by_title': recommend_by_title_keyword
    }, f)

print("✓ Keyword-based method saved successfully!")
print(f"  - Similarity: keyword_similarity.npy")
print(f"  - Functions: keyword_recommender_funcs.pkl")

In [ ]:
# Save preprocessed data for evaluation
# Save the processed dataframe with all features
data_for_eval = data[[
    'title', 'description', 'type_of_food', 'calories', 'cook_time',
    'ingredients', 'step', 'source',
    'title_clean', 'description_clean', 'step_clean', 'combined_text',
    'ingredients_text', 'ingredients_clean', 'keywords',
    'cook_time_minutes', 'calories_numeric'
]].copy()

data_for_eval.to_csv('../notebooks/saved_models/preprocessed_data.csv', index=True, encoding='utf-8-sig')
print("✓ Preprocessed data saved: preprocessed_data.csv")

# Create a summary file with information about all methods
summary = {
    'num_recipes': len(data),
    'methods': {
        'tfidf': {
            'description': 'Text similarity using TF-IDF on title + description + steps',
            'similarity_matrix_shape': tfidf_similarity.shape,
            'files': ['tfidf_vectorizer.pkl', 'tfidf_matrix.pkl', 'tfidf_similarity.npy']
        },
        'ingredient_tfidf': {
            'description': 'TF-IDF on ingredient text for better ingredient matching',
            'similarity_matrix_shape': ingredient_tfidf_similarity.shape,
            'files': ['ingredient_tfidf_vectorizer.pkl', 'ingredient_tfidf_matrix.pkl', 'ingredient_tfidf_similarity.npy']
        },
        'keyword': {
            'description': 'Jaccard similarity on extracted keywords from recipe titles',
            'similarity_matrix_shape': keyword_similarity_matrix.shape,
            'files': ['keyword_similarity.npy']
        },
        'hybrid': {
            'description': 'Weighted combination: TF-IDF (0.4) + Ingredient TF-IDF (0.6)',
            'similarity_matrix_shape': hybrid_similarity_matrix.shape,
            'files': ['hybrid_similarity.npy']
        }
    }
}

import json
with open('../notebooks/saved_models/methods_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False, default=str)

print("✓ Methods summary saved: methods_summary.json")
print("\n" + "="*80)
print("ALL CONTENT-BASED METHODS SAVED SUCCESSFULLY!")
print("="*80)
print(f"\nSaved to: ../notebooks/saved_models/")
print(f"\nFiles created:")
print("  - preprocessed_data.csv")
print("  - methods_summary.json")
print("  - TF-IDF: 3 files")
print("  - Ingredient TF-IDF: 3 files")
print("  - Keyword: 1 file")
print("  - Hybrid: 1 file")
print(f"\nTotal: 12 files ready for evaluation")

## 6. Save All Data for Evaluation

Lưu preprocessed data và tất cả models/similarity matrices để phục vụ cho đánh giá